In [1]:
# ============================================
# 1. Install Dependencies
# ============================================

!pip -q install sentence-transformers
!pip -q install python-docx
!pip -q install rapidfuzz
!pip -q install tqdm

In [3]:
# ============================================
# 2. Import Libraries
# ============================================

import json
import re
import heapq

import numpy as np
import pandas as pd

from tqdm import tqdm
from docx import Document

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

In [4]:
# ============================================
# 3. Load Job Description
# ============================================

def load_job_description(path):

    doc = Document(path)

    paragraphs = []

    for paragraph in doc.paragraphs:

        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return "\n".join(paragraphs)


job_description = load_job_description("job_description.docx")

print("Job Description Loaded Successfully!")

Job Description Loaded Successfully!


In [5]:
# ============================================
# 4. Candidate Generator
# ============================================

def candidate_generator(path):


    with open(path, "r", encoding="utf-8") as file:

        for line in file:

            if line.strip():

                yield json.loads(line)

In [6]:
# ============================================
# 5. Text Processing
# ============================================

def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^a-z0-9+#./ ]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def candidate_to_text(candidate):

    profile = candidate.get("profile", {})

    parts = []

    parts.append(
        profile.get("current_title", "")
    )

    parts.append(
        profile.get("headline", "")
    )

    parts.append(
        profile.get("summary", "")
    )


    for job in candidate.get(
        "career_history",
        []
    ):

        parts.append(
            job.get("title", "")
        )

        parts.append(
            job.get(
                "description",
                ""
            )[:200]
        )

    skills = [

        skill.get("name", "")

        for skill in candidate.get(
            "skills",
            []
        )

    ]

    parts.append(
        " ".join(skills)
    )

    return "\n".join(parts)


def candidate_to_semantic_text(candidate):

    profile = candidate.get("profile", {})

    parts = []

    parts.append(
        profile.get("current_title", "")
    )

    parts.append(
        profile.get("headline", "")
    )

    parts.append(
        profile.get(
            "summary",
            ""
        )[:900]
    )


    for job in candidate.get(
        "career_history",
        []
    )[:2]:

        parts.append(
            job.get("title", "")
        )

        parts.append(
            job.get(
                "description",
                ""
            )[:200]
        )

    # Skills last

    skills = [

        skill.get("name", "")

        for skill in candidate.get(
            "skills",
            []
        )

    ]

    parts.append(
        " ".join(skills)
    )

    return "\n".join(parts)


print("Text Processing Functions Ready!")

Text Processing Functions Ready!


In [7]:
# ============================================
# 6. Parse Job Description
# ============================================

TECH_KEYWORDS = [

    "python",
    "transformers",
    "llm",
    "rag",
    "langchain",
    "embeddings",
    "faiss",
    "pinecone",
    "milvus",
    "qdrant",
    "weaviate",
    "elasticsearch",
    "opensearch",
    "lora",
    "qlora",
    "peft",
    "nlp",
    "machine learning",
    "deep learning",
    "sentence transformers",
    "vector database",
    "retrieval",
    "ranking",
    "recommendation"

]

SOFT_SKILLS = [

    "startup",
    "mentor",
    "product",
    "communication",
    "leadership",
    "collaboration"

]

jd_text = clean_text(job_description)

jd_skills = [
    skill
    for skill in TECH_KEYWORDS
    if skill in jd_text
]

jd_soft_skills = [
    skill
    for skill in SOFT_SKILLS
    if skill in jd_text
]

JD_PROFILE = {

    "skills": jd_skills,
    "soft_skills": jd_soft_skills

}

print("JD Parsed Successfully!")

JD Parsed Successfully!


In [8]:
# ============================================
# 7. Feature Engineering
# ============================================

JD_SKILL_WEIGHTS = {

    "embeddings": 10,
    "retrieval": 10,
    "ranking": 10,
    "vector database": 10,

    "faiss": 9,
    "pinecone": 9,
    "milvus": 9,
    "qdrant": 9,
    "weaviate": 9,

    "elasticsearch": 8,
    "opensearch": 8,

    "llm": 8,
    "rag": 8,
    "sentence transformers": 8,
    "langchain": 7,
    "transformers": 7,

    "lora": 6,
    "qlora": 6,
    "peft": 6,

    "python": 5,
    "nlp": 5,
    "recommendation": 5

}

# ============================================
# Skill Aliases
# ============================================

SKILL_ALIASES = {

    "embeddings": [
        "embedding",
        "embeddings",
        "sentence transformers",
        "sentence-transformers",
        "openai embeddings",
        "bge",
        "e5"
    ],

    "retrieval": [
        "retrieval",
        "information retrieval",
        "semantic search",
        "vector search"
    ],

    "ranking": [
        "ranking",
        "learning to rank",
        "learning-to-rank",
        "ltr"
    ],

    "llm": [
        "llm",
        "large language model",
        "large language models",
        "fine tuning llms",
        "fine-tuning llms"
    ],

    "rag": [
        "rag",
        "retrieval augmented generation",
        "retrieval-augmented generation"
    ],

    "vector database": [
        "vector database",
        "vector databases",
        "vector db",
        "pgvector"
    ],

    "recommendation": [
        "recommendation",
        "recommendation systems",
        "recommender systems",
        "recsys"
    ],

    "nlp": [
        "nlp",
        "natural language processing"
    ]

}


def jd_skill_score(candidate):

    # ---------- Career text ----------
    career_text = " ".join(

        (
            job.get("title", "")
            + " "
            + job.get("description", "")
        ).lower()

        for job in candidate.get(
            "career_history",
            []
        )

    )

    # ---------- Skills text ----------
    skills_text = " ".join(

        skill.get("name", "").lower()

        for skill in candidate.get(
            "skills",
            []
        )

    )

    score = 0
    matched = []

    for skill, weight in JD_SKILL_WEIGHTS.items():

        aliases = SKILL_ALIASES.get(
            skill,
            [skill]
        )

        # Prefer evidence from career history
        if any(
            alias in career_text
            for alias in aliases
        ):

            score += weight
            matched.append(skill)
            continue

        # Lowered weight for declared skills
        if any(
            alias in skills_text
            for alias in aliases
        ):

            score += weight * 0.5
            matched.append(skill)

    return score, matched


def experience_score(years):

    if 5 <= years <= 9:
        return 10

    elif years < 5:

        return max(
            0,
            10 - (5 - years) * 2
        )

    else:

        return max(
            0,
            10 - (years - 9)
        )


def behavior_score(signals):

    score = 0

    if signals.get(
        "open_to_work_flag",
        False
    ):
        score += 5

    score += (
        signals.get(
            "recruiter_response_rate",
            0
        ) * 5
    )

    score += (
        signals.get(
            "interview_completion_rate",
            0
        ) * 5
    )

    score += (
        signals.get(
            "github_activity_score",
            0
        ) / 20
    )

    score += (
        signals.get(
            "saved_by_recruiters_30d",
            0
        ) / 5
    )

    score -= (
        signals.get(
            "notice_period_days",
            0
        ) / 30
    )

    return score

from datetime import datetime

REFERENCE_DATE = datetime(2026, 6, 1)

def activity_score(signals):

    last_active = signals.get("last_active_date")

    if not last_active:
        return 0

    try:
        last_active = datetime.strptime(
            last_active,
            "%Y-%m-%d"
        )
    except:
        return 0

    days = (REFERENCE_DATE - last_active).days

    if days <= 30:
        return 3

    elif days <= 90:
        return 2

    elif days <= 180:
        return 0

    else:
        return -5

PRODUCTION_WORDS = {

    "production",
    "pipeline",
    "deployment",
    "serving",
    "monitoring",
    "evaluation",
    "latency",
    "distributed",
    "scalable",
    "inference",

    "semantic search",
    "vector search",
    "information retrieval",
    "query expansion",
    "nearest neighbor",
    "nearest neighbors",

    "ab testing",
    "drift",
    "retraining",
    "observability"

}


def production_score(candidate):

    text = candidate.get("_cached_text")

    if text is None:

        text = candidate_to_text(candidate).lower()

    return sum(

        word in text

        for word in PRODUCTION_WORDS

    )

# ============================================
# AI Title Signals
# ============================================

AI_TITLE_KEYWORDS = [

    "ai",
    "ml",
    "machine learning",
    "nlp",
    "recommendation",
    "search",
    "retrieval",
    "applied scientist"

]


def title_score(candidate):

    title = candidate.get(
        "profile",
        {}
    ).get(
        "current_title",
        ""
    ).lower()

    return sum(

        keyword in title

        for keyword in AI_TITLE_KEYWORDS

    )

# ============================================
# Relevant AI Experience
# ============================================

AI_ROLE_KEYWORDS = {

    "ai",
    "artificial intelligence",
    "machine learning",
    "ml",
    "nlp",
    "llm",
    "rag",
    "retrieval",
    "ranking",
    "search",
    "recommendation",
    "recommender",
    "applied scientist",
    "machine learning engineer",
    "ml engineer",
    "ai engineer",
    "data scientist"

}


def relevant_ai_experience(candidate):

    total_months = 0

    for job in candidate.get("career_history", []):

        text = (
            job.get("title", "") + " " +
            job.get("description", "")
        ).lower()

        if any(
            keyword in text
            for keyword in AI_ROLE_KEYWORDS
        ):

            total_months += job.get(
                "duration_months",
                0
            )

    if total_months == 0:

        return candidate.get(
            "profile",
            {}
        ).get(
            "years_of_experience",
            0
        )

    return total_months / 12

# ============================================
# Consulting Career Penalty
# ============================================

CONSULTING_COMPANIES = {

    "tcs",
    "tata consultancy services",
    "infosys",
    "wipro",
    "cognizant",
    "capgemini",
    "accenture",
    "tech mahindra",
    "hcl",
    "hcl technologies",
    "ibm consulting",
    "ltimindtree",
    "mindtree",
    "l&t technology services",
    "ltts"

}


def consulting_penalty(candidate):

    jobs = candidate.get("career_history", [])

    if len(jobs) == 0:
        return 0

    consulting_jobs = 0

    for job in jobs:

        company = job.get(
            "company",
            ""
        ).lower()

        if any(
            firm in company
            for firm in CONSULTING_COMPANIES
        ):
            consulting_jobs += 1

    if consulting_jobs == len(jobs):
        return -15

    return 0

# ============================================
# Fast Candidate Score
# ============================================

def fast_score(candidate):

    profile = candidate.get(
        "profile",
        {}
    )

    skill_score, _ = jd_skill_score(candidate)

    relevant_years = relevant_ai_experience(candidate)

    exp_score = experience_score(
      relevant_years
    )

    behaviour = behavior_score(

        candidate.get(
            "redrob_signals",
            {}
        )

    )
    behaviour += activity_score(
    candidate["redrob_signals"]
)

    production = production_score(candidate)

    title = title_score(candidate)

    consulting = consulting_penalty(candidate)

    return (

        skill_score * 7

        + exp_score * 2

        + behaviour

        + production * 4

        + title * 3

        + consulting

    )

In [9]:
# ============================================
# 8. Candidate Retrieval
# ============================================

TOP_PRESELECT = 500

heap = []

for candidate in tqdm(candidate_generator("candidates.jsonl")):

    candidate["_cached_text"] = candidate_to_text(candidate).lower()

    score = fast_score(candidate)

    item = (
        score,
        candidate["candidate_id"],
        candidate
    )

    if len(heap) < TOP_PRESELECT:

        heapq.heappush(
            heap,
            item
        )

    else:

        heapq.heappushpop(
            heap,
            item
        )

preselected = sorted(
    heap,
    reverse=True
)

print(f"Retrieved {len(preselected)} candidates.")

100000it [00:37, 2684.65it/s]

Retrieved 500 candidates.


In [10]:
# ============================================
# 9. Load Embedding Model
# ============================================

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding Model Loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding Model Loaded!


In [12]:
# ============================================
# 10. Semantic Embeddings
# ============================================

semantic_jd = """
Senior AI Engineer.

Applied machine learning.

Production AI systems.

Production ML deployment.

Embeddings.

Embeddings-based retrieval.

Dense retrieval.

Hybrid retrieval.

Semantic search.

Vector search.

Candidate retrieval.

Candidate matching.

Search ranking.

Learning to Rank.

Recommendation systems.

Production recommendation systems.

Vector databases.

FAISS.

Pinecone.

Milvus.

Qdrant.

Weaviate.

OpenSearch.

Elasticsearch.

Sentence Transformers.

Python.

Natural Language Processing.

Large Language Models.

Retrieval-Augmented Generation.

LoRA.

QLoRA.

PEFT.

Evaluation frameworks.

NDCG.

MRR.

MAP.

Offline evaluation.

Online evaluation.

A/B testing.

Embedding drift.

Index refresh.

Monitoring.

Model serving.

Scalable inference.

Distributed ML systems.

Product engineering.

Product company experience.

End-to-end ML systems.

Production search infrastructure.

Production ranking systems.

Recruiter search.

Recruiter workflows.

Behavioral signals.

Candidate availability.

Production-ready AI systems.
"""

jd_embedding = model.encode(

    semantic_jd,

    normalize_embeddings=True

)

def candidate_to_semantic_text(candidate):

    profile = candidate.get("profile", {})

    parts = []

    # Current role
    parts.append(
        profile.get("current_title", "")
    )

    # Headline
    parts.append(
        profile.get("headline", "")
    )

    # Longer summary
    parts.append(
        profile.get("summary", "")[:200]
    )

    # Career history
    for job in candidate.get(
        "career_history",
        []
    )[:2]:

        parts.append(
            job.get("title", "")
        )

        parts.append(
            job.get(
                "description",
                ""
            )[:300]
        )

    # Skills last
    skills = [

        skill.get("name", "")

        for skill in candidate.get(
            "skills",
            []
        )

    ]

    parts.append(
        " ".join(skills)
    )

    return "\n".join(parts)


candidate_texts = [

    candidate_to_semantic_text(candidate)

    for _, _, candidate in preselected

]

candidate_embeddings = model.encode(

    candidate_texts,

    batch_size=128,

    normalize_embeddings=True,

    show_progress_bar=True

)

semantic_scores = cosine_similarity(

    [jd_embedding],

    candidate_embeddings

)[0]

print("Semantic Ranking Ready!")

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
# ============================================
# 11. Final Ranking
# ============================================

fast_scores = np.array([x[0] for x in preselected])

fast_min = fast_scores.min()
fast_max = fast_scores.max()

final_rankings = []

for semantic, (fast, cid, candidate) in zip(
    semantic_scores,
    preselected
):

    fast_norm = (
        fast - fast_min
    ) / (
        fast_max - fast_min + 1e-9
    )

    final_score = (
        0.60 * fast_norm +
        0.40 * semantic
    )

    final_rankings.append(
        (
            final_score,
            cid,
            candidate,
            semantic,
            fast
        )
    )

final_rankings.sort(reverse=True)

print("Final Ranking Complete!")

Final Ranking Complete!


In [14]:
# ============================================
# 12. Reason Generation
# ============================================

def generate_reason(candidate, semantic_score):

    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})

    title = profile.get(
        "current_title",
        "Unknown"
    )

    exp = profile.get(
        "years_of_experience",
        0
    )

    _, matched = jd_skill_score(candidate)

    matched = sorted(
        matched,
        key=lambda x: JD_SKILL_WEIGHTS.get(x, 0),
        reverse=True
    )[:5]

    DISPLAY_NAMES = {

        "embeddings": "Embeddings",
        "retrieval": "Retrieval",
        "ranking": "Ranking",
        "vector database": "Vector Databases",
        "faiss": "FAISS",
        "pinecone": "Pinecone",
        "milvus": "Milvus",
        "qdrant": "Qdrant",
        "weaviate": "Weaviate",
        "elasticsearch": "Elasticsearch",
        "opensearch": "OpenSearch",
        "llm": "LLMs",
        "rag": "RAG",
        "sentence transformers": "Sentence Transformers",
        "transformers": "Transformers",
        "python": "Python",
        "nlp": "NLP",
        "recommendation": "Recommendation Systems",
        "lora": "LoRA",
        "qlora": "QLoRA",
        "peft": "PEFT"

    }

    matched = [
        DISPLAY_NAMES.get(skill, skill)
        for skill in matched
    ]

    if len(matched) == 0:

        skill_text = "modern AI systems"

    elif len(matched) == 1:

        skill_text = matched[0]

    elif len(matched) == 2:

        skill_text = (
            matched[0]
            + " and "
            + matched[1]
        )

    else:

        skill_text = (
            ", ".join(matched[:-1])
            + " and "
            + matched[-1]
        )

    reason = (
        f"{title} with {exp:.1f} yrs experience"
    )

    # Mentioned notice period only when it is a concern
    notice = signals.get(
        "notice_period_days",
        0
    )

    if notice >= 90:

        reason += (
            f"; {notice}-day notice period offset by strong technical alignment"
        )

    # Mentoned experience only when far outside JD range
    if exp > 12:

        reason += (
            "; experience exceeds the preferred range but closely aligns with the role"
        )

    elif exp < 5:

        reason += (
            "; slightly below the preferred experience range"
        )

    return reason + "."

In [15]:
# ============================================
# 13. Generate Submission
# ============================================

submission_rows = []

for rank, (
    score,
    cid,
    candidate,
    semantic,
    fast
) in enumerate(
    final_rankings[:100],
    start=1
):

    submission_rows.append({

        "Candidate_id": cid,

        "Rank": rank,

        "Score": round(
            float(score),
            4
        ),

        "Reasoning": generate_reason(
            candidate,
            semantic
        )

    })

submission = pd.DataFrame(
    submission_rows,
    columns=[
        "Candidate_id",
        "Rank",
        "Score",
        "Reasoning"
    ]
)

# Ensures correct ordering
submission = submission.sort_values(
    "Rank"
).reset_index(
    drop=True
)

print(f"Generated {len(submission)} ranked candidates.")

submission.head(10)

Generated 100 ranked candidates.


,Candidate_id,Rank,Score,Reasoning
0,CAND_0039754,1,0.9441,Senior Applied Scientist with 16.2 yrs experie...
1,CAND_0030953,2,0.9146,Search Engineer with 7.8 yrs experience.
2,CAND_0046064,3,0.9023,Senior NLP Engineer with 8.9 yrs experience.
3,CAND_0079387,4,0.8857,AI Engineer with 6.9 yrs experience.
4,CAND_0083307,5,0.8841,Search Engineer with 7.8 yrs experience; 120-d...
5,CAND_0007009,6,0.8776,Recommendation Systems Engineer with 7.9 yrs e...
6,CAND_0044855,7,0.8483,Senior Data Scientist with 6.6 yrs experience.
7,CAND_0077337,8,0.8472,Staff Machine Learning Engineer with 7.0 yrs e...
8,CAND_0055992,9,0.8452,AI Engineer with 16.9 yrs experience; experien...
9,CAND_0041611,10,0.8366,Staff Machine Learning Engineer with 6.4 yrs e...


In [16]:
# ============================================
# 14. Save Submission
# ============================================

submission.to_csv(
    "Submission.csv",
    index=False
)

print("Submission Saved Successfully!")

Submission Saved Successfully!


In [18]:
# ============================================
# 15. Validate & Download
# ============================================

print(submission.info())

print()

print(submission.head())

from google.colab import files

files.download("Submission.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Candidate_id  100 non-null    object 
 1   Rank          100 non-null    int64  
 2   Score         100 non-null    float64
 3   Reasoning     100 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 3.3+ KB
None

   Candidate_id  Rank   Score  \
0  CAND_0039754     1  0.9441   
1  CAND_0030953     2  0.9146   
2  CAND_0046064     3  0.9023   
3  CAND_0079387     4  0.8857   
4  CAND_0083307     5  0.8841   

                                           Reasoning  
0  Senior Applied Scientist with 16.2 yrs experie...  
1           Search Engineer with 7.8 yrs experience.  
2       Senior NLP Engineer with 8.9 yrs experience.  
3               AI Engineer with 6.9 yrs experience.  
4  Search Engineer with 7.8 yrs experience; 120-d...  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>